# Package a Customer H2O Model

This is the intake gate for a customer native H2O binary or MOJO ZIP. We are not training, converting, importing, or scoring the model in the local notebook environment. We verify the artifact, schema, optional golden fixtures, and target-runtime contract before creating anything in Azure.

## Before you run it

The repository starts with an independently published H2O-3 prostate GBM MOJO under `data/h2o/customer_bundle/demo`. The defaults in `.env.example` select that demo so you can learn the workflow before handling private customer artifacts.

To replace the demo, follow `data/h2o/customer_bundle/README.md`: create an ignored `private/<bundle-id>/` directory, place these three files together, and replace the complete `H2O_CUSTOMER_*` contract in `.env`:

- a native binary created with `h2o.save_model()` or a MOJO ZIP created with `model.download_mojo()`
- a golden input CSV with columns in scoring order
- a golden expected CSV with one `predict` value for each input row

The root `.venv` runs this packaging notebook and does not need the model producer's H2O version. Notebook 03 builds the Azure ML environment from the declared target H2O, Python, and Java versions. Notebook 04 is the first step that imports the MOJO and proves golden parity.

> Native binaries require the producer H2O version at runtime. MOJOs may be scored by a separately selected compatible runtime. When golden fixtures are supplied, parity is checked in that runtime before optional traffic promotion.

We identify the model format automatically, inspect `model.ini` for MOJOs, validate optional golden-file structure, and write a checksum manifest. Nothing is uploaded to Azure in this notebook.

**Source:** Adapted from this repository's H2O reference and onboarding notebooks.

In [3]:
from pathlib import Path
import hashlib
import json
import os
import sys

import pandas as pd
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

load_dotenv(WORKSHOP_ROOT / ".env", override=True)

def workshop_path(name: str) -> Path:
    value = Path(os.environ[name])
    return value if value.is_absolute() else WORKSHOP_ROOT / value

def optional_workshop_path(name: str, legacy_name: str | None = None) -> Path | None:
    value = os.getenv(name, "").strip()
    if not value and legacy_name:
        value = os.getenv(legacy_name, "").strip()
    if not value:
        return None
    path = Path(value)
    return path if path.is_absolute() else WORKSHOP_ROOT / path

def enabled(name: str) -> bool:
    return os.getenv(name, "false").lower() in {"1", "true", "yes"}

sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from validate_bundle import detect_model_format, read_mojo_metadata, validate_bundle

print(f"Workshop root: {WORKSHOP_ROOT}")

Workshop root: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop


## 1. Read the customer contract

The values in `workshop/.env` are our agreement about how this model must run. We resolve the three bundle paths, require immutable asset versions, and check the feature lists before reading the model.

Review the table printed by the next cell. If a version, target, or feature is wrong, stop here and correct `.env`; do not adjust the model to make the notebook pass.

In [4]:
MODEL_PATH = workshop_path("H2O_CUSTOMER_MODEL_PATH").resolve()
GOLDEN_INPUT_PATH = optional_workshop_path(
    "H2O_CUSTOMER_GOLDEN_INPUT_PATH",
    "H2O_CUSTOMER_INPUT_PATH",
)
GOLDEN_EXPECTED_PATH = optional_workshop_path(
    "H2O_CUSTOMER_GOLDEN_EXPECTED_PATH",
    "H2O_CUSTOMER_EXPECTED_PATH",
)
if (GOLDEN_INPUT_PATH is None) != (GOLDEN_EXPECTED_PATH is None):
    raise ValueError("Provide both customer golden paths or leave both blank")
GOLDEN_PROVIDED = GOLDEN_INPUT_PATH is not None
RUN_GOLDEN_VALIDATION = GOLDEN_PROVIDED and os.getenv(
    "H2O_CUSTOMER_RUN_GOLDEN_VALIDATION",
    "true",
).lower() in {"1", "true", "yes"}
REQUIRE_GOLDEN_VALIDATION = enabled("H2O_CUSTOMER_REQUIRE_GOLDEN_VALIDATION")
if REQUIRE_GOLDEN_VALIDATION and not GOLDEN_PROVIDED:
    raise ValueError("Golden validation is required but no golden files are configured")
if REQUIRE_GOLDEN_VALIDATION and not RUN_GOLDEN_VALIDATION:
    raise ValueError("Required golden validation cannot be disabled")

BUNDLE_DIR = MODEL_PATH.parent
DEMO_BUNDLE_DIR = (
    WORKSHOP_ROOT / "data/h2o/customer_bundle/demo"
).resolve()
BUNDLE_SOURCE = (
    "bundled demo" if BUNDLE_DIR == DEMO_BUNDLE_DIR else "customer upload"
)

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Customer model not found: {MODEL_PATH}")
detected_model_format = detect_model_format(MODEL_PATH)
configured_model_format = os.getenv("H2O_CUSTOMER_MODEL_FORMAT", "auto").strip().lower()
if configured_model_format not in {"auto", "h2o_binary", "h2o_mojo"}:
    raise ValueError("H2O_CUSTOMER_MODEL_FORMAT must be auto, h2o_binary, or h2o_mojo")
if configured_model_format != "auto" and configured_model_format != detected_model_format:
    raise ValueError(
        f"Configured model format {configured_model_format} does not match "
        f"detected format {detected_model_format}"
    )
MODEL_FORMAT = detected_model_format

configured_model_h2o_version = os.getenv(
    "H2O_CUSTOMER_MODEL_H2O_VERSION",
    os.getenv("H2O_CUSTOMER_VERSION", ""),
).strip()
configured_mojo_version = os.getenv("H2O_CUSTOMER_MOJO_VERSION", "").strip()
mojo_metadata = read_mojo_metadata(MODEL_PATH) if MODEL_FORMAT == "h2o_mojo" else None
if mojo_metadata:
    MODEL_H2O_VERSION = mojo_metadata["h2o_version"]
    MOJO_VERSION = mojo_metadata["mojo_version"]
    if configured_model_h2o_version and configured_model_h2o_version != MODEL_H2O_VERSION:
        raise ValueError("H2O_CUSTOMER_MODEL_H2O_VERSION does not match model.ini")
    if configured_mojo_version and configured_mojo_version != MOJO_VERSION:
        raise ValueError("H2O_CUSTOMER_MOJO_VERSION does not match model.ini")
else:
    MODEL_H2O_VERSION = configured_model_h2o_version
    MOJO_VERSION = None
    if not MODEL_H2O_VERSION:
        raise ValueError("Native H2O binaries require H2O_CUSTOMER_MODEL_H2O_VERSION")

RUNTIME_H2O_VERSION = os.getenv(
    "H2O_CUSTOMER_RUNTIME_VERSION",
    os.getenv("H2O_CUSTOMER_VERSION", ""),
).strip()
if MODEL_FORMAT == "h2o_binary" and RUNTIME_H2O_VERSION != MODEL_H2O_VERSION:
    raise ValueError("Native H2O binaries require matching producer and runtime versions")
PYTHON_VERSION = os.environ["H2O_CUSTOMER_PYTHON_VERSION"].strip()
JAVA_VERSION = os.environ["H2O_CUSTOMER_JAVA_VERSION"].strip()
H2O_PIP_SPEC = (
    os.getenv("H2O_CUSTOMER_PIP_SPEC", "").strip()
    or f"h2o=={RUNTIME_H2O_VERSION}"
)
FEATURES = [
    value.strip()
    for value in os.getenv("H2O_CUSTOMER_FEATURES", "").split(",")
    if value.strip()
]
if mojo_metadata:
    if FEATURES and FEATURES != mojo_metadata["features"]:
        raise ValueError(f"Customer features must match MOJO columns: {mojo_metadata['features']}")
    FEATURES = mojo_metadata["features"]
CATEGORICAL_FEATURES = [
    value.strip()
    for value in os.getenv("H2O_CUSTOMER_CATEGORICAL_FEATURES", "").split(",")
    if value.strip()
]
TARGET = os.getenv("H2O_CUSTOMER_TARGET", "").strip()
if mojo_metadata:
    if TARGET and TARGET != mojo_metadata["target"]:
        raise ValueError(f"Customer target must match MOJO response: {mojo_metadata['target']}")
    TARGET = mojo_metadata["target"]
MODEL_NAME = os.environ["H2O_CUSTOMER_MODEL_NAME"].strip()
MODEL_VERSION = os.environ["H2O_CUSTOMER_MODEL_VERSION"].strip()

required_contract = {
    "H2O_CUSTOMER_RUNTIME_VERSION": RUNTIME_H2O_VERSION,
    "H2O_CUSTOMER_PYTHON_VERSION": PYTHON_VERSION,
    "H2O_CUSTOMER_JAVA_VERSION": JAVA_VERSION,
    "H2O_CUSTOMER_FEATURES": FEATURES,
    "H2O_CUSTOMER_TARGET": TARGET,
    "H2O_CUSTOMER_MODEL_NAME": MODEL_NAME,
    "H2O_CUSTOMER_MODEL_VERSION": MODEL_VERSION,
}

missing = [name for name, value in required_contract.items() if not value]
if missing:
    raise ValueError("Missing customer model contract values: " + ", ".join(missing))
if MODEL_VERSION.lower() == "latest":
    raise ValueError("Use an immutable H2O model version, not 'latest'")
if len(FEATURES) != len(set(FEATURES)):
    raise ValueError("Customer features must not contain duplicates")
if not set(CATEGORICAL_FEATURES).issubset(FEATURES):
    raise ValueError(
        "Categorical features must be a subset of H2O_CUSTOMER_FEATURES"
    )

display(
    pd.DataFrame(
        {
            "setting": [
                "bundle source",
                "bundle directory",
                "model file",
                "model",
                "format",
                "producer H2O",
                "MOJO format",
                "runtime H2O",
                "Python",
                "Java",
                "target",
                "features",
                "categorical features",
                "golden data",
                "run golden validation",
                "golden validation required",
            ],
            "value": [
                BUNDLE_SOURCE,
                str(BUNDLE_DIR),
                MODEL_PATH.name,
                f"{MODEL_NAME}:{MODEL_VERSION}",
                MODEL_FORMAT,
                MODEL_H2O_VERSION,
                MOJO_VERSION or "not applicable",
                RUNTIME_H2O_VERSION,
                PYTHON_VERSION,
                JAVA_VERSION,
                TARGET,
                ", ".join(FEATURES),
                ", ".join(CATEGORICAL_FEATURES) or "none",
                "provided" if GOLDEN_PROVIDED else "not provided",
                RUN_GOLDEN_VALIDATION,
                REQUIRE_GOLDEN_VALIDATION,
            ],
        }
    )
)

FileNotFoundError: Customer model not found: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/data/h2o/customer_bundle/model

## 2. Check the files and golden data

The model is required. Golden input and expected files are optional, but they must be provided together and colocated with the model. MOJO metadata is read directly from `model.ini`; native binaries use the declared producer version and schema. No H2O process is started.

When golden files are present, the input CSV must use the exact feature order and the expected CSV must have one `predict` column with the same number of rows. With no golden files, packaging and deployment remain available unless `H2O_CUSTOMER_REQUIRE_GOLDEN_VALIDATION=true`.

In [ ]:
golden_input = None
golden_expected = None
prediction_type = None
if GOLDEN_PROVIDED:
    for path in (GOLDEN_INPUT_PATH, GOLDEN_EXPECTED_PATH):
        if not path.is_file():
            raise FileNotFoundError(f"Customer golden input not found: {path}")
        if path.parent != BUNDLE_DIR:
            raise ValueError("The model and golden files must be colocated")

    golden_input = pd.read_csv(GOLDEN_INPUT_PATH)
    golden_expected = pd.read_csv(GOLDEN_EXPECTED_PATH)
    if list(golden_input.columns) != FEATURES:
        raise ValueError(f"Golden input columns must be ordered as: {FEATURES}")
    if list(golden_expected.columns) != ["predict"]:
        raise ValueError("Golden expected must contain one column named 'predict'")
    if golden_input.empty or len(golden_expected) != len(golden_input):
        raise ValueError("Golden files must contain the same non-zero row count")

    expected_numeric = pd.to_numeric(golden_expected["predict"], errors="coerce")
    prediction_type = "number" if expected_numeric.notna().all() else "string"
    print(f"Golden rows: {len(golden_input)}")
    print(f"Prediction type: {prediction_type}")
    display(golden_input.head())
    display(golden_expected.head())
else:
    print("Golden data is not provided; runtime parity will be skipped.")

## 3. Write the model manifest

The manifest is the handoff record for this bundle. It keeps the runtime, schema, prediction type, and SHA-256 checksum for every supplied file together with the model.

Packaging status is `passed` after structural checks and checksums succeed. Runtime validation is `pending` when golden files are present and `not_provided` otherwise. Notebook 04 performs optional parity in the registered Azure ML environment.

In [ ]:
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "model_format": MODEL_FORMAT,
    "h2o_version": MODEL_H2O_VERSION,
    "runtime_h2o_version": RUNTIME_H2O_VERSION,
    "h2o_pip_spec": H2O_PIP_SPEC,
    "python_version": PYTHON_VERSION,
    "java_version": JAVA_VERSION,
    "model_file": MODEL_PATH.name,
    "target": TARGET,
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "prediction_type": prediction_type,
    "files": {MODEL_PATH.name: sha256(MODEL_PATH)},
    "golden_data": {
        "provided": GOLDEN_PROVIDED,
        "validate": RUN_GOLDEN_VALIDATION,
        "required": REQUIRE_GOLDEN_VALIDATION,
        "input_file": GOLDEN_INPUT_PATH.name if GOLDEN_PROVIDED else None,
        "expected_file": GOLDEN_EXPECTED_PATH.name if GOLDEN_PROVIDED else None,
    },
    "packaging": {"status": "passed"},
    "validation": {
        "status": (
            "pending"
            if GOLDEN_PROVIDED and RUN_GOLDEN_VALIDATION
            else "skipped"
            if GOLDEN_PROVIDED
            else "not_provided"
        ),
        "location": "target_runtime",
    },
}
if MODEL_FORMAT == "h2o_mojo":
    manifest["mojo_version"] = MOJO_VERSION
if GOLDEN_PROVIDED:
    manifest["files"].update(
        {
            GOLDEN_INPUT_PATH.name: sha256(GOLDEN_INPUT_PATH),
            GOLDEN_EXPECTED_PATH.name: sha256(GOLDEN_EXPECTED_PATH),
        }
    )
if BUNDLE_SOURCE == "bundled demo":
    manifest["source"] = {
        "repository": "https://github.com/h2oai/h2o-3",
        "revision": "ce08492c71e6f2fe9ce902cd82b1e7828ca3b1f6",
        "path": (
            "h2o-genmodel/src/test/resources/hex/genmodel/algos/gbm/"
            "gbm_variable_importance.zip"
        ),
        "license": "Apache-2.0",
    }

manifest_path = BUNDLE_DIR / "model_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)

print(f"Manifest written to: {manifest_path}")
display(manifest)

## 4. Validate the bundle structure

This check reads the manifest back from disk and verifies checksums, detected format, selected runtime, schema, optional golden files, and MOJO metadata when applicable.

In [ ]:
bundle_summary = validate_bundle(
    BUNDLE_DIR,
    RUNTIME_H2O_VERSION,
    require_packaging_validation=True,
)
display(bundle_summary)

## 5. Defer prediction parity to the target runtime

Packaging is complete. Notebook 03 creates the customer-specific Azure ML environment. When golden files are present, notebook 04 deploys with zero traffic and compares predictions before optional promotion. When they are absent, validation is skipped unless explicitly required.

This separation lets the shared workshop `.venv` package native binaries and MOJO ZIPs produced by different H2O versions without mutating the local kernel.

In [ ]:
print("Packaging validation passed.")
print(f"Target runtime: h2o=={RUNTIME_H2O_VERSION}, Python {PYTHON_VERSION}, Java {JAVA_VERSION}")
print(
    "Runtime golden parity remains pending until notebook 04."
    if GOLDEN_PROVIDED and RUN_GOLDEN_VALIDATION
    else "Runtime golden parity is skipped by configuration."
    if GOLDEN_PROVIDED
    else "Runtime golden parity is optional and no golden data was provided."
)

## Expected Result

The selected demo or private customer folder contains a validated dual-format manifest. No H2O package or JVM was required locally; optional runtime parity is handled in notebook 04.

Next: `02_register_model.ipynb`.